# k-TSP — el problema **correcto** (seleccionar exactamente *k* nodos)

## Qué es el k-TSP (y por qué la versión anterior estaba mal)

El **k-TSP** que aparece en la página de LKH-3 de Helsgaun y que define el paper que se citaba
(**Pandiri & Singh, OPSEARCH 57:1164-1204, 2020**; y antes Venkatesh, Srivastava & Singh, 2018) es:

> Dado un conjunto de **n** ciudades que incluye una **ciudad-hogar**, y un valor fijo **k** con
> **1 < k ≤ n**, encontrar un tour de **longitud mínima** que parte y termina en la ciudad-hogar y
> visita **exactamente k ciudades** (incluyendo la hogar) de entre las n. Resolverlo implica
> **(1)** elegir el subconjunto de k ciudades (con la hogar dentro) y **(2)** ordenarlas en un tour.

Es decir:

- **Hay UN solo viajante.**
- La **"k" es el número de nodos a seleccionar**, NO el número de viajantes.
- Se visitan **exactamente k** ciudades; las otras **n − k se dejan fuera**.
- **Objetivo: minimizar la distancia TOTAL** del único tour (NO min-max).

### Por qué la versión anterior no cuadraba

La versión anterior resolvía otra cosa: **k viajantes** que se reparten **todas** las n ciudades,
minimizando la ruta más larga (**min-max m-TSP**). Eso es el problema `mTSP` de LKH-3 con
`MTSP_OBJECTIVE = MINMAX`, **no** el `k-TSP`. Por eso los BKS "no cuadraban": se comparaban
resultados de un problema contra valores de otro. Esta versión corrige la definición de raíz.

| | k-TSP (correcto, esta versión) | min-max m-TSP (versión anterior) |
|---|---|---|
| Viajantes | 1 | k |
| Qué es k | nodos a seleccionar | nº de viajantes |
| Ciudades visitadas | exactamente k de n | todas las n |
| Objetivo | minimizar distancia total | minimizar ruta más larga |

## Qué hace este notebook

1. Descarga instancias TSPLIB y construye la matriz de distancias respetando `EDGE_WEIGHT_TYPE`.
2. Resuelve el k-TSP verdadero con tres enfoques heurísticos en Python (selección de k nodos +
   ordenamiento + 2-opt): un **constructivo voraz**, un **GRASP/multi-start** y un **algoritmo genético**.
3. Genera los archivos para el **binario oficial LKH-3** en el formato del k-TSP
   (`SPECIAL` + `K = k`), de modo que puedas obtener el óptimo/BKS real ejecutándolo.
4. Tabla comparativa con GAP contra el mejor valor conocido (el del binario LKH-3 si lo corres,
   o el mejor de las heurísticas en caso contrario).

## Valores de k: los tres cuartiles oficiales

El benchmark oficial del k-TSP (paquete de Helsgaun, basado en Pandiri & Singh) define **tres
valores de k por instancia**, correspondientes a tres cuartiles del tamaño n:

- **Q1: k ≈ n/4** (conjunto TSPLIB_1)
- **Q2: k ≈ n/2** (conjunto TSPLIB_2)
- **Q3: k ≈ 3n/4** (conjunto TSPLIB_3)

Este notebook ejecuta **los tres cuartiles** (`K_QUARTILES = [1, 2, 3]`) y compara cada resultado
contra el **mejor valor conocido (BKS) oficial** de Helsgaun para ese cuartil. Produce un CSV
combinado (`resultados_kTSP.csv`) y uno por cuartil (`resultados_kTSP_Q1/2/3.csv`), además de un
resumen de GAP por método y por cuartil. Así se evalúa el comportamiento de las heurísticas a
medida que k crece (de visitar pocos nodos a visitar casi todos).

## NBR — heurística propia (Nicolás Barrientos Results)

**NBR (Nicolás Barrientos Results)** es la heurística propia de este trabajo. Está diseñada para
ser **simple, explicable y replicable**, no una caja negra:

- **Criterio único y claro:** selecciona los k nodos por *costo marginal de inserción* mínimo
  (cuánto crece el tour al añadir cada nodo en su mejor posición). Así decide a la vez **qué
  nodos** visitar y **en qué orden**, que son las dos decisiones propias del k-TSP.
- **Refinamiento:** 2-opt sobre el ciclo + intercambio de un nodo dentro del tour por uno fuera
  si reduce la distancia total, iterando hasta no mejorar.
- **Determinista y replicable:** NBR **no usa azar**. Con los mismos datos produce siempre el
  mismo tour, por lo que el resultado es 100% reproducible (los empates se rompen por índice de
  nodo). Esto es deliberado: prioriza ser correcta y verificable sobre ganar por suerte.
- **Resuelve el k-TSP correcto:** entrega un único tour que visita exactamente k nodos
  (incluida la ciudad-hogar) y minimiza la distancia total.

> **Nota honesta sobre los BKS:** la versión anterior traía una tabla de "BKS oficiales" que en
> realidad eran de min-max m-TSP. Aquí **no** se hardcodean esos valores. El BKS se toma del
> binario LKH-3 oficial si lo ejecutas (Paso 1); si no, se usa el mejor de las heurísticas como
> referencia provisional, claramente etiquetada como tal. Así los números **sí** cuadran con el
> problema que se está resolviendo.


## Paso 1 — (opcional) Instalar el binario oficial LKH-3

Esta celda descarga **LKH-3.0.14**, lo compila y deja la ruta en `LKH_EXECUTABLE_PATH`.
El binario sí resuelve el k-TSP de forma (cuasi-)óptima usando el keyword `K`.
Si no lo ejecutas, el notebook funciona igual con las heurísticas en Python.

In [ ]:
# Descarga y compila el binario oficial LKH-3 (opcional pero recomendado).
import os, subprocess, shutil, urllib.request, tarfile
from pathlib import Path

LKH_VERSION = "3.0.14"
LKH_TGZ_URLS = [
    f"http://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-{LKH_VERSION}.tgz",
    f"http://akira.ruc.dk/~keld/research/LKH-3/LKH-{LKH_VERSION}.tgz",
]
LKH_HOME   = Path(f"/content/LKH-{LKH_VERSION}") if Path("/content").exists() else Path(f"./LKH-{LKH_VERSION}").resolve()
LKH_BINARY = LKH_HOME / "LKH"
LKH_TGZ    = Path(f"./LKH-{LKH_VERSION}.tgz").resolve()

def _run(cmd, **kw):
    print("  $ " + " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, check=True, **kw)

def setup_lkh3() -> str:
    if LKH_BINARY.exists() and os.access(LKH_BINARY, os.X_OK):
        print(f"[OK] LKH-3 ya compilado en: {LKH_BINARY}")
        return str(LKH_BINARY)
    if shutil.which("gcc") is None or shutil.which("make") is None:
        print("[INFO] Instalando gcc/make…")
        try:
            _run(["apt-get", "update", "-qq"])
            _run(["apt-get", "install", "-y", "-qq", "build-essential"])
        except Exception as e:
            print(f"[WARN] No se pudo instalar build-essential: {e}")
    if not LKH_TGZ.exists():
        last_err = None
        for url in LKH_TGZ_URLS:
            try:
                print(f"[INFO] Descargando {url}")
                urllib.request.urlretrieve(url, LKH_TGZ)
                print(f"[OK] Descargado: {LKH_TGZ} ({LKH_TGZ.stat().st_size//1024} KB)")
                last_err = None
                break
            except Exception as e:
                last_err = e
                print(f"[WARN] Falló {url}: {e}")
        if last_err is not None:
            print("[ERROR] No se pudo descargar LKH-3. Hazlo a mano:")
            for url in LKH_TGZ_URLS: print("   wget " + url)
            return ""
    print("[INFO] Extrayendo…")
    with tarfile.open(LKH_TGZ, "r:gz") as tar:
        tar.extractall(LKH_HOME.parent)
    print("[INFO] Compilando (make)…")
    try:
        _run(["make"], cwd=str(LKH_HOME))
    except Exception as e:
        print(f"[ERROR] make falló: {e}")
        return ""
    if LKH_BINARY.exists():
        print(f"[OK] LKH-3 compilado: {LKH_BINARY}")
        return str(LKH_BINARY)
    print("[ERROR] No se encontró el binario tras compilar.")
    return ""

LKH_EXECUTABLE_PATH = setup_lkh3()
print("\n" + "=" * 60)
if LKH_EXECUTABLE_PATH:
    print("LISTO. La celda principal usará automáticamente:")
    print(f"    LKH_EXECUTABLE = \"{LKH_EXECUTABLE_PATH}\"")
else:
    print("Sin binario LKH-3: el notebook usará solo las heurísticas en Python.")
print("=" * 60)


In [ ]:
!pip install numpy pandas requests tabulate -q

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  k-TSP VERDADERO  —  seleccionar exactamente k nodos, 1 tour, min distancia ║
# ║  Definición: Pandiri & Singh (2020); Venkatesh, Srivastava & Singh (2018). ║
# ║  k = número de NODOS a visitar (incluida la ciudad-hogar), NO viajantes.   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os, re, gzip, json, math, time, random, hashlib, subprocess
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import requests

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ─── CONFIGURACIÓN ────────────────────────────────────────────────────────────
SEED  = 42
HOME  = 0                 # ciudad-hogar (depósito). Siempre incluida en el tour.

# ─── VALOR DE k (nº de nodos a visitar, incluida la ciudad-hogar) ─────────────
# IMPORTANTE: el k-TSP NO tiene un k "canónico" único; depende de lo que se pida.
# Mientras confirmas con tu profesor qué k espera, el repositorio trae un valor
# por DEFECTO funcional para que el código corra tal cual: k = round(0.50 * n)
# (visitar la mitad de las ciudades). Cuando tengas el k oficial, cámbialo abajo.
#
# Modos disponibles:
#   "official" -> usa el k OFICIAL de Helsgaun/Pandiri-Singh (recomendado). El cuartil
#                 se elige con K_QUARTILE: 1 -> k~n/4, 2 -> k~n/2, 3 -> k~3n/4.
#                 Además permite comparar contra el BKS oficial real (GAP correcto).
#   "ratio"    -> k = round(K_RATIO * n)
#   "fixed"    -> k = K_FIXED para todas las instancias
#   "list"     -> usa K_PER_INSTANCE (cae a "ratio" si la instancia no aparece)
K_MODE      = "official"  # <- RECOMENDADO: reproduce el benchmark oficial del k-TSP
# Cuartiles a ejecutar cuando K_MODE == "official". El benchmark oficial define
# tres valores de k por instancia: Q1 (k~n/4), Q2 (k~n/2), Q3 (k~3n/4).
# Por defecto se ejecutan LOS TRES. Para uno solo, deja p.ej. K_QUARTILES = [1].
K_QUARTILES = [1, 2, 3]
K_QUARTILE  = 1           # <- compatibilidad: se sobrescribe en el bucle de cuartiles
K_RATIO    = 0.50         # <- solo si K_MODE == "ratio"
K_FIXED    = 20           # <- solo si K_MODE == "fixed"
K_PER_INSTANCE: Dict[str, int] = {}   # <- solo si K_MODE == "list"

# ─── DATOS OFICIALES DEL k-TSP (Helsgaun, LKH-3) ──────────────────────────────
# k y mejor solución conocida (BKS) por instancia y cuartil, extraídos del paquete
# oficial k-TSP de Keld Helsgaun (fuente: Pandiri & Singh, OPSEARCH 2020).
# Permiten (a) fijar el k correcto y (b) comparar contra el óptimo/BKS real.
OFFICIAL_KTSP = {
    "burma14": {"n": 14, "k_q1": 3, "bks_q1": 359, "k_q2": 7, "bks_q2": 1272, "k_q3": 10, "bks_q3": 1642},
    "ulysses16": {"n": 16, "k_q1": 4, "bks_q1": 935, "k_q2": 8, "bks_q2": 1685, "k_q3": 12, "bks_q3": 3183},
    "gr17": {"n": 17, "k_q1": 4, "bks_q1": 234, "k_q2": 8, "bks_q2": 517, "k_q3": 12, "bks_q3": 951},
    "gr21": {"n": 21, "k_q1": 5, "bks_q1": 324, "k_q2": 10, "bks_q2": 918, "k_q3": 15, "bks_q3": 1501},
    "ulysses22": {"n": 22, "k_q1": 5, "bks_q1": 747, "k_q2": 11, "bks_q2": 1902, "k_q3": 16, "bks_q3": 2941},
    "gr24": {"n": 24, "k_q1": 6, "bks_q1": 264, "k_q2": 12, "bks_q2": 504, "k_q3": 18, "bks_q3": 844},
    "fri26": {"n": 26, "k_q1": 6, "bks_q1": 243, "k_q2": 13, "bks_q2": 414, "k_q3": 19, "bks_q3": 601},
    "bayg29": {"n": 29, "k_q1": 7, "bks_q1": 332, "k_q2": 14, "bks_q2": 626, "k_q3": 21, "bks_q3": 999},
    "bays29": {"n": 29, "k_q1": 7, "bks_q1": 400, "k_q2": 14, "bks_q2": 733, "k_q3": 21, "bks_q3": 1194},
    "dantzig42": {"n": 42, "k_q1": 10, "bks_q1": 145, "k_q2": 21, "bks_q2": 260, "k_q3": 31, "bks_q3": 427},
    "swiss42": {"n": 42, "k_q1": 10, "bks_q1": 192, "k_q2": 21, "bks_q2": 458, "k_q3": 31, "bks_q3": 760},
    "att48": {"n": 48, "k_q1": 12, "bks_q1": 1925, "k_q2": 24, "bks_q2": 3603, "k_q3": 36, "bks_q3": 6563},
    "gr48": {"n": 48, "k_q1": 12, "bks_q1": 874, "k_q2": 24, "bks_q2": 1819, "k_q3": 36, "bks_q3": 3104},
    "hk48": {"n": 48, "k_q1": 12, "bks_q1": 2827, "k_q2": 24, "bks_q2": 4701, "k_q3": 36, "bks_q3": 7278},
    "eil51": {"n": 51, "k_q1": 12, "bks_q1": 82, "k_q2": 25, "bks_q2": 175, "k_q3": 38, "bks_q3": 287},
    "berlin52": {"n": 52, "k_q1": 13, "bks_q1": 679, "k_q2": 26, "bks_q2": 1874, "k_q3": 39, "bks_q3": 4174},
    "brazil58": {"n": 58, "k_q1": 14, "bks_q1": 4965, "k_q2": 29, "bks_q2": 7978, "k_q3": 43, "bks_q3": 11614},
    "st70": {"n": 70, "k_q1": 17, "bks_q1": 120, "k_q2": 35, "bks_q2": 260, "k_q3": 52, "bks_q3": 428},
    "eil76": {"n": 76, "k_q1": 19, "bks_q1": 102, "k_q2": 38, "bks_q2": 213, "k_q3": 57, "bks_q3": 335},
    "pr76": {"n": 76, "k_q1": 19, "bks_q1": 23450, "k_q2": 38, "bks_q2": 41248, "k_q3": 57, "bks_q3": 64142},
    "gr96": {"n": 96, "k_q1": 24, "bks_q1": 10460, "k_q2": 48, "bks_q2": 20688, "k_q3": 72, "bks_q3": 31343},
    "rat99": {"n": 99, "k_q1": 24, "bks_q1": 284, "k_q2": 49, "bks_q2": 573, "k_q3": 74, "bks_q3": 857},
    "kroA100": {"n": 100, "k_q1": 25, "bks_q1": 4931, "k_q2": 50, "bks_q2": 9184, "k_q3": 75, "bks_q3": 14379},
    "kroB100": {"n": 100, "k_q1": 25, "bks_q1": 4305, "k_q2": 50, "bks_q2": 9096, "k_q3": 75, "bks_q3": 14680},
    "kroC100": {"n": 100, "k_q1": 25, "bks_q1": 4964, "k_q2": 50, "bks_q2": 9457, "k_q3": 75, "bks_q3": 13892},
    "kroD100": {"n": 100, "k_q1": 25, "bks_q1": 4762, "k_q2": 50, "bks_q2": 8719, "k_q3": 75, "bks_q3": 14058},
    "kroE100": {"n": 100, "k_q1": 25, "bks_q1": 3905, "k_q2": 50, "bks_q2": 9097, "k_q3": 75, "bks_q3": 14622},
    "rd100": {"n": 100, "k_q1": 25, "bks_q1": 1438, "k_q2": 50, "bks_q2": 3168, "k_q3": 75, "bks_q3": 5094},
    "eil101": {"n": 101, "k_q1": 25, "bks_q1": 107, "k_q2": 50, "bks_q2": 227, "k_q3": 75, "bks_q3": 386},
    "lin105": {"n": 105, "k_q1": 26, "bks_q1": 2606, "k_q2": 52, "bks_q2": 5848, "k_q3": 78, "bks_q3": 8983},
    "pr107": {"n": 107, "k_q1": 26, "bks_q1": 8443, "k_q2": 53, "bks_q2": 18028, "k_q3": 80, "bks_q3": 36292},
    "gr120": {"n": 120, "k_q1": 30, "bks_q1": 1308, "k_q2": 60, "bks_q2": 2669, "k_q3": 90, "bks_q3": 4325},
    "pr124": {"n": 124, "k_q1": 31, "bks_q1": 14325, "k_q2": 62, "bks_q2": 22998, "k_q3": 93, "bks_q3": 39008},
    "bier127": {"n": 127, "k_q1": 31, "bks_q1": 10619, "k_q2": 63, "bks_q2": 25761, "k_q3": 95, "bks_q3": 49680},
    "ch130": {"n": 130, "k_q1": 32, "bks_q1": 1130, "k_q2": 65, "bks_q2": 2408, "k_q3": 97, "bks_q3": 3876},
    "pr136": {"n": 136, "k_q1": 34, "bks_q1": 21116, "k_q2": 68, "bks_q2": 46414, "k_q3": 102, "bks_q3": 67910},
    "gr137": {"n": 137, "k_q1": 34, "bks_q1": 17399, "k_q2": 68, "bks_q2": 29231, "k_q3": 102, "bks_q3": 43670},
    "pr144": {"n": 144, "k_q1": 36, "bks_q1": 14327, "k_q2": 72, "bks_q2": 27424, "k_q3": 108, "bks_q3": 41194},
    "ch150": {"n": 150, "k_q1": 37, "bks_q1": 1276, "k_q2": 75, "bks_q2": 2760, "k_q3": 112, "bks_q3": 4366},
    "kroA150": {"n": 150, "k_q1": 37, "bks_q1": 5690, "k_q2": 75, "bks_q2": 11496, "k_q3": 112, "bks_q3": 17395},
    "kroB150": {"n": 150, "k_q1": 37, "bks_q1": 5812, "k_q2": 75, "bks_q2": 11357, "k_q3": 112, "bks_q3": 16879},
    "pr152": {"n": 152, "k_q1": 38, "bks_q1": 20029, "k_q2": 76, "bks_q2": 36333, "k_q3": 114, "bks_q3": 51826},
    "u159": {"n": 159, "k_q1": 39, "bks_q1": 8983, "k_q2": 79, "bks_q2": 18323, "k_q3": 119, "bks_q3": 27096},
    "si175": {"n": 175, "k_q1": 43, "bks_q1": 4873, "k_q2": 87, "bks_q2": 10087, "k_q3": 131, "bks_q3": 15460},
    "brg180": {"n": 180, "k_q1": 45, "bks_q1": 500, "k_q2": 90, "bks_q2": 980, "k_q3": 135, "bks_q3": 1480},
    "rat195": {"n": 195, "k_q1": 48, "bks_q1": 554, "k_q2": 97, "bks_q2": 1100, "k_q3": 146, "bks_q3": 1667},
    "d198": {"n": 198, "k_q1": 49, "bks_q1": 4964, "k_q2": 99, "bks_q2": 6921, "k_q3": 148, "bks_q3": 9093},
    "kroA200": {"n": 200, "k_q1": 50, "bks_q1": 6154, "k_q2": 100, "bks_q2": 12372, "k_q3": 150, "bks_q3": 19211},
    "kroB200": {"n": 200, "k_q1": 50, "bks_q1": 6016, "k_q2": 100, "bks_q2": 12338, "k_q3": 150, "bks_q3": 18898},
    "gr202": {"n": 202, "k_q1": 50, "bks_q1": 8142, "k_q2": 101, "bks_q2": 13790, "k_q3": 151, "bks_q3": 20927},
    "ts225": {"n": 225, "k_q1": 56, "bks_q1": 28828, "k_q2": 112, "bks_q2": 56828, "k_q3": 168, "bks_q3": 85656},
    "tsp225": {"n": 225, "k_q1": 56, "bks_q1": 913, "k_q2": 112, "bks_q2": 1708, "k_q3": 168, "bks_q3": 2624},
    "pr226": {"n": 226, "k_q1": 56, "bks_q1": 20033, "k_q2": 113, "bks_q2": 36720, "k_q3": 169, "bks_q3": 47142},
    "gr229": {"n": 229, "k_q1": 57, "bks_q1": 18555, "k_q2": 114, "bks_q2": 39618, "k_q3": 171, "bks_q3": 64457},
    "gil262": {"n": 262, "k_q1": 65, "bks_q1": 541, "k_q2": 131, "bks_q2": 993, "k_q3": 196, "bks_q3": 1551},
    "pr264": {"n": 264, "k_q1": 66, "bks_q1": 9232, "k_q2": 132, "bks_q2": 22644, "k_q3": 198, "bks_q3": 34346},
    "a280": {"n": 280, "k_q1": 70, "bks_q1": 649, "k_q2": 140, "bks_q2": 1231, "k_q3": 210, "bks_q3": 1845},
    "pr299": {"n": 299, "k_q1": 74, "bks_q1": 11077, "k_q2": 149, "bks_q2": 22390, "k_q3": 224, "bks_q3": 32738},
    "lin318": {"n": 318, "k_q1": 79, "bks_q1": 8544, "k_q2": 159, "bks_q2": 15916, "k_q3": 238, "bks_q3": 25303},
    "rd400": {"n": 400, "k_q1": 100, "bks_q1": 3091, "k_q2": 200, "bks_q2": 6350, "k_q3": 300, "bks_q3": 9972},
    "fl417": {"n": 417, "k_q1": 104, "bks_q1": 1831, "k_q2": 208, "bks_q2": 5436, "k_q3": 312, "bks_q3": 7930},
    "gr431": {"n": 431, "k_q1": 107, "bks_q1": 14525, "k_q2": 215, "bks_q2": 34985, "k_q3": 323, "bks_q3": 73921},
    "pr439": {"n": 439, "k_q1": 109, "bks_q1": 19751, "k_q2": 219, "bks_q2": 35404, "k_q3": 329, "bks_q3": 58006},
    "pcb442": {"n": 442, "k_q1": 110, "bks_q1": 10932, "k_q2": 221, "bks_q2": 22537, "k_q3": 331, "bks_q3": 34358},
    "d493": {"n": 493, "k_q1": 123, "bks_q1": 9356, "k_q2": 246, "bks_q2": 13574, "k_q3": 369, "bks_q3": 20931},
    "att532": {"n": 532, "k_q1": 133, "bks_q1": 3812, "k_q2": 266, "bks_q2": 8978, "k_q3": 399, "bks_q3": 16360},
    "ali535": {"n": 535, "k_q1": 133, "bks_q1": 11773, "k_q2": 267, "bks_q2": 42563, "k_q3": 401, "bks_q3": 89316},
    "si535": {"n": 535, "k_q1": 133, "bks_q1": 10839, "k_q2": 267, "bks_q2": 22327, "k_q3": 401, "bks_q3": 34320},
    "pa561": {"n": 561, "k_q1": 140, "bks_q1": 465, "k_q2": 280, "bks_q2": 1035, "k_q3": 420, "bks_q3": 1745},
    "u574": {"n": 574, "k_q1": 143, "bks_q1": 7584, "k_q2": 287, "bks_q2": 14858, "k_q3": 430, "bks_q3": 23616},
    "rat575": {"n": 575, "k_q1": 143, "bks_q1": 1572, "k_q2": 287, "bks_q2": 3120, "k_q3": 431, "bks_q3": 4730},
    "p654": {"n": 654, "k_q1": 163, "bks_q1": 6929, "k_q2": 327, "bks_q2": 16174, "k_q3": 490, "bks_q3": 21446},
    "d657": {"n": 657, "k_q1": 164, "bks_q1": 11116, "k_q2": 328, "bks_q2": 20946, "k_q3": 492, "bks_q3": 32119},
    "gr666": {"n": 666, "k_q1": 166, "bks_q1": 26381, "k_q2": 333, "bks_q2": 76925, "k_q3": 499, "bks_q3": 146122},
    "u724": {"n": 724, "k_q1": 181, "bks_q1": 8709, "k_q2": 362, "bks_q2": 16799, "k_q3": 543, "bks_q3": 27469},
    "rat783": {"n": 783, "k_q1": 195, "bks_q1": 2004, "k_q2": 391, "bks_q2": 4008, "k_q3": 587, "bks_q3": 6160},
}

def official_k_and_bks(name: str, n: int):
    """Devuelve (k_oficial, bks_oficial) para el cuartil K_QUARTILE, o (None, None)."""
    d = OFFICIAL_KTSP.get(name)
    if not d:
        return None, None
    return d.get(f"k_q{K_QUARTILE}"), d.get(f"bks_q{K_QUARTILE}")

RUN_SMALL, RUN_MEDIUM, RUN_LARGE = True, True, False
MAX_INSTANCES = None      # None = todas las seleccionadas; o un entero

# LKH-3 DESACTIVADO por defecto: la referencia es el BKS OFICIAL de Helsgaun
# (ya incorporado en OFFICIAL_KTSP), que es óptimo/near-óptimo y NO requiere
# ejecutar el binario. Ejecutar LKH localmente solo añade horas sin mejorar la
# referencia. Si AÚN así quieres correr el binario, pon: USE_LKH_BINARY = True
USE_LKH_BINARY = False
LKH_EXECUTABLE = globals().get("LKH_EXECUTABLE_PATH", "") if USE_LKH_BINARY else ""

# Presupuestos de tiempo de las heurísticas (segundos)
GREEDY_RESTARTS = 5
GRASP_TIME      = 3.0
GA_TIME         = 4.0

# Parámetros del binario LKH-3 (k-TSP usa SPECIAL + K)
LKH3_MAX_TRIALS = 10000
LKH3_RUNS       = 10

BASE_DIR    = Path(".")
TSPLIB_DIR  = BASE_DIR / "tsplib"
RESULTS_DIR = BASE_DIR / "resultados_ktsp"
TSPLIB_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ─── INSTANCIAS TSPLIB ────────────────────────────────────────────────────────
TSPLIB_SOURCES = [
    "https://raw.githubusercontent.com/mastqe/tsplib/master/{name}.tsp",
    "https://softlib.rice.edu/pub/tsplib/tsp/{name}.tsp.gz",
    "http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/{name}.tsp.gz",
]
TSPLIB_SIZES: Dict[str, int] = {
    "att48": 48, "eil51": 51, "berlin52": 52, "st70": 70, "pr76": 76, "eil76": 76,
    "kroA100": 100, "kroB100": 100, "kroC100": 100, "kroD100": 100, "kroE100": 100,
    "eil101": 101, "lin105": 105, "pr107": 107, "gr120": 120, "pr124": 124,
    "bier127": 127, "ch130": 130, "pr136": 136, "gr137": 137, "ch150": 150,
    "kroA150": 150, "kroB150": 150, "rat195": 195, "kroA200": 200, "kroB200": 200,
    "ts225": 225, "tsp225": 225, "pr226": 226, "gr229": 229, "gil262": 262,
    "pr264": 264, "a280": 280, "pr299": 299, "lin318": 318, "rd400": 400,
    "fl417": 417, "pcb442": 442, "pr439": 439,
    "att532": 532, "rat575": 575, "p654": 654, "d657": 657, "gr666": 666,
}

def size_bucket(n: int) -> str:
    if n <= 150: return "small"
    if n <= 500: return "medium"
    return "large"

def choose_k(name: str, n: int) -> int:
    if K_MODE == "official":
        k_off, _ = official_k_and_bks(name, n)
        k = k_off if k_off is not None else int(round(0.25 * n))
    elif K_MODE == "list" and name in K_PER_INSTANCE:
        k = K_PER_INSTANCE[name]
    elif K_MODE == "fixed":
        k = K_FIXED
    else:
        k = int(round(K_RATIO * n))
    # k-TSP exige 1 < k <= n
    return max(2, min(n, k))

SELECTED: List[Tuple[str, int]] = []
for _name, _n in TSPLIB_SIZES.items():
    b = size_bucket(_n)
    if (b == "small" and RUN_SMALL) or (b == "medium" and RUN_MEDIUM) or (b == "large" and RUN_LARGE):
        SELECTED.append((_name, _n))
SELECTED.sort(key=lambda t: (t[1], t[0]))
if MAX_INSTANCES is not None:
    SELECTED = SELECTED[:MAX_INSTANCES]

# ─── DESCARGA Y PARSER TSPLIB ─────────────────────────────────────────────────
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download_tsplib_instance(name: str) -> Tuple[Path, str]:
    out = TSPLIB_DIR / (name + ".tsp")
    if out.exists() and out.stat().st_size > 0:
        return out, "cached_local_file"
    errors = []
    for tpl in TSPLIB_SOURCES:
        url = tpl.format(name=name)
        try:
            r = requests.get(url, timeout=30)
            if r.status_code != 200 or len(r.content) < 50:
                errors.append(f"{url} -> status {r.status_code}"); continue
            data = r.content
            if url.endswith(".gz"):
                data = gzip.decompress(data)
            out.write_bytes(data)
            return out, url
        except Exception as e:
            errors.append(f"{url} -> {e!r}")
    raise RuntimeError(f"No se pudo descargar {name}.tsp. Errores: {errors}")

def parse_tsplib_tsp(path: Path) -> Tuple[np.ndarray, str, Dict[str, str]]:
    text = path.read_text(errors="ignore").splitlines()
    meta: Dict[str, str] = {}
    coords, in_coords = [], False
    for raw in text:
        line = raw.strip()
        if not line: continue
        if ":" in line and not in_coords:
            k, v = line.split(":", 1); meta[k.strip().upper()] = v.strip()
        if line.upper().startswith("NODE_COORD_SECTION"):
            in_coords = True; continue
        if in_coords:
            if line.upper().startswith(("EOF", "DISPLAY_DATA_SECTION", "DEPOT_SECTION")): break
            parts = line.split()
            if len(parts) >= 3:
                try: coords.append((int(float(parts[0])), float(parts[1]), float(parts[2])))
                except ValueError: pass
    if not coords:
        raise ValueError(f"Sin coordenadas en {path}")
    coords.sort(key=lambda z: z[0])
    arr = np.array([[x, y] for _, x, y in coords], dtype=float)
    ewtype = meta.get("EDGE_WEIGHT_TYPE", "EUC_2D").upper().replace(" ", "_")
    return arr, ewtype, meta

def nint(x: float) -> int: return int(x + 0.5)
def geo_to_rad(x: float) -> float:
    deg = int(x); minutes = x - deg
    return math.pi * (deg + 5.0 * minutes / 3.0) / 180.0

def tsplib_distance_matrix(coords: np.ndarray, ewtype: str) -> np.ndarray:
    n = len(coords); D = np.zeros((n, n)); typ = ewtype.upper()
    for i in range(n):
        for j in range(n):
            if i == j: continue
            dx = coords[i,0]-coords[j,0]; dy = coords[i,1]-coords[j,1]
            if typ == "EUC_2D":   D[i,j] = nint(math.hypot(dx, dy))
            elif typ == "CEIL_2D": D[i,j] = math.ceil(math.hypot(dx, dy))
            elif typ == "ATT":
                rij = math.sqrt((dx*dx+dy*dy)/10.0); tij = nint(rij)
                D[i,j] = tij if tij >= rij else tij+1
            elif typ == "GEO":
                RRR = 6378.388
                la_i, lo_i = geo_to_rad(coords[i,0]), geo_to_rad(coords[i,1])
                la_j, lo_j = geo_to_rad(coords[j,0]), geo_to_rad(coords[j,1])
                q1 = math.cos(lo_i-lo_j); q2 = math.cos(la_i-la_j); q3 = math.cos(la_i+la_j)
                D[i,j] = int(RRR*math.acos(0.5*((1+q1)*q2-(1-q1)*q3))+1.0)
            else:
                raise NotImplementedError(f"EDGE_WEIGHT_TYPE no soportado: {ewtype}")
    return D

# ─── k-TSP: utilidades vectorizadas (1 tour, k nodos, min distancia total) ────
def tour_length(tour, D: np.ndarray) -> float:
    """Longitud del ciclo cerrado sobre los k nodos (vectorizado)."""
    t = np.asarray(tour)
    return float(D[t, np.roll(t, -1)].sum())

def validate_ktsp(tour, n: int, k: int, home: int = HOME):
    if len(tour) != k:               return False, f"|tour|={len(tour)} != k={k}"
    if home not in tour:             return False, "la ciudad-hogar no está en el tour"
    if len(set(tour)) != len(tour):  return False, "nodos repetidos"
    if any(not (0 <= v < n) for v in tour): return False, "nodo fuera de rango"
    return True, "OK"

def two_opt_cycle(tour, D, max_passes: int = 40):
    """2-opt vectorizado por NumPy sobre el ciclo (mantiene el subconjunto)."""
    best = list(tour); m = len(best)
    if m < 4: return best
    improved = True; passes = 0
    while improved and passes < max_passes:
        improved = False; passes += 1
        arr = np.asarray(best); nxt = np.roll(arr, -1); d_ab = D[arr, nxt]
        for i in range(m - 1):
            a = arr[i]; b = arr[i + 1]
            js = np.arange(i + 2, m)
            if i == 0:
                js = js[js != m - 1]
            if js.size == 0: continue
            c = arr[js]; d = arr[(js + 1) % m]
            delta = D[a, c] + D[b, d] - d_ab[i] - d_ab[js]
            jmin = int(np.argmin(delta))
            if delta[jmin] < -1e-9:
                j = int(js[jmin])
                best[i + 1:j + 1] = best[i + 1:j + 1][::-1]
                improved = True
                arr = np.asarray(best); nxt = np.roll(arr, -1); d_ab = D[arr, nxt]
    return best

def swap_in_out(tour, D, n, home=HOME):
    """Intercambio dentro/fuera vectorizado: por posición prueba todos los nodos fuera."""
    inset = np.zeros(n, dtype=bool); inset[np.asarray(tour)] = True
    outside = np.where(~inset)[0]
    if outside.size == 0: return tour, False
    best = list(tour); m = len(best); improved = False
    for pos in range(m):
        node = best[pos]
        if node == home: continue
        a = best[pos - 1]; b = best[(pos + 1) % m]
        cur = D[a, node] + D[node, b]
        cand_cost = D[a, outside] + D[outside, b]
        cmin = int(np.argmin(cand_cost))
        if cand_cost[cmin] + 1e-9 < cur:
            new = int(outside[cmin])
            best[pos] = new
            inset[node] = False; inset[new] = True
            outside = np.where(~inset)[0]
            improved = True
    return best, improved

def local_search_ktsp(tour, D, n, home=HOME, max_rounds=8):
    cur = two_opt_cycle(tour, D)
    for _ in range(max_rounds):
        cur, ch = swap_in_out(cur, D, n, home)
        cur = two_opt_cycle(cur, D)
        if not ch: break
    return cur

def _cheapest_insertion_build(D, n, k, home, seed_node, rng=None, cand_limit=80):
    """Construye un tour de k nodos por inserción más barata desde [home, seed_node]."""
    tour = [home, seed_node]
    inset = np.zeros(n, dtype=bool); inset[home] = True; inset[seed_node] = True
    while len(tour) < k:
        outside = np.where(~inset)[0]
        if outside.size == 0: break
        cand = outside
        if rng is not None and outside.size > cand_limit:
            cand = np.array(rng.sample(list(outside), cand_limit))
        arr = np.asarray(tour); nxt = np.roll(arr, -1)
        # delta[c, e] = D[a_e,c] + D[c,b_e] - D[a_e,b_e]
        A = arr; B = nxt
        dab = D[A, B]                              # (m,)
        dac = D[np.ix_(cand, A)]                    # (|cand|, m)
        dcb = D[np.ix_(cand, B)]
        delta = dac + dcb - dab[None, :]            # (|cand|, m)
        flat = int(np.argmin(delta))
        ci, ei = divmod(flat, delta.shape[1])
        node = int(cand[ci]); pos = ei + 1
        tour.insert(pos, node); inset[node] = True
    return tour

# ─── HEURÍSTICA 1: constructivo voraz (cheapest insertion vectorizado) ────────
def greedy_ktsp(D, n, k, home=HOME, restarts=GREEDY_RESTARTS, seed=SEED):
    rng = random.Random(seed)
    best_tour, best_len = None, float("inf")
    order = np.argsort(D[home])
    seeds = [int(v) for v in order if v != home][:max(1, restarts)]
    for s in seeds:
        tour = _cheapest_insertion_build(D, n, k, home, s)
        tour = local_search_ktsp(tour, D, n, home)
        L = tour_length(tour, D)
        if L < best_len:
            best_tour, best_len = tour, L
    return {"tour": best_tour, "length": best_len, "source": "greedy_insertion+LS"}

# ─── HEURÍSTICA 2: GRASP / multi-start ────────────────────────────────────────
def grasp_ktsp(D, n, k, home=HOME, time_limit=GRASP_TIME, seed=SEED):
    rng = random.Random(seed + 1)
    best_tour, best_len = None, float("inf")
    others = [v for v in range(n) if v != home]
    t0 = time.time()
    while time.time() - t0 < time_limit:
        # construcción aleatorizada: nodos cercanos a la hogar con ruido
        noise = np.array([rng.random() for _ in range(len(others))])
        score = D[home, others] * (0.7 + 0.6 * noise)
        idx = np.argsort(score)
        chosen = [home] + [others[i] for i in idx[:k - 1]]
        tour = local_search_ktsp(chosen, D, n, home)
        L = tour_length(tour, D)
        if L < best_len:
            best_tour, best_len = tour, L
    if best_tour is None:  # fallback si el budget fue ~0
        best_tour = local_search_ktsp([home] + others[:k-1], D, n, home)
        best_len = tour_length(best_tour, D)
    return {"tour": best_tour, "length": best_len, "source": "GRASP_multistart+LS"}

# ─── HEURÍSTICA 3: algoritmo genético (subconjunto + orden) ───────────────────
def ga_ktsp(D, n, k, home=HOME, pop_size=30, time_limit=GA_TIME, seed=SEED):
    rng = random.Random(seed + 2)
    others = [v for v in range(n) if v != home]
    def random_indiv():
        return [home] + rng.sample(others, k - 1)
    def fitness(ind): return tour_length(ind, D)
    def crossover(p1, p2):
        keep = set([home]) | set(rng.sample(p1[1:], (k - 1) // 2))
        child = [home] + [v for v in p1[1:] if v in keep]
        for v in p2[1:]:
            if len(child) >= k: break
            if v not in keep: child.append(v); keep.add(v)
        pool = [v for v in others if v not in keep]; rng.shuffle(pool)
        while len(child) < k: child.append(pool.pop())
        return child
    def mutate(ind, pm=0.3):
        ind = ind[:]
        if rng.random() < pm:
            i, j = rng.sample(range(1, k), 2); ind[i], ind[j] = ind[j], ind[i]
        if rng.random() < pm:
            inset = set(ind); outside = [v for v in others if v not in inset]
            if outside:
                pos = rng.randrange(1, k); ind[pos] = rng.choice(outside)
        return ind
    # población inicial: algunos con LS, el resto aleatorios (para no gastar todo el budget)
    pop = [random_indiv() for _ in range(pop_size)]
    pop[0] = local_search_ktsp(pop[0], D, n, home)
    pop.sort(key=fitness)
    t0 = time.time()
    while time.time() - t0 < time_limit:
        new = pop[:max(2, pop_size // 5)]
        while len(new) < pop_size:
            a = min(rng.sample(pop, 3), key=fitness)
            b = min(rng.sample(pop, 3), key=fitness)
            child = mutate(crossover(a, b))
            if rng.random() < 0.25:
                child = local_search_ktsp(child, D, n, home)
            new.append(child)
        pop = sorted(new, key=fitness)[:pop_size]
    best = local_search_ktsp(pop[0], D, n, home)  # pulido final del mejor
    return {"tour": best, "length": tour_length(best, D), "source": "GA_subset+order+LS"}

# ─── HEURÍSTICA PROPIA: DCIH (Nicolás Barrientos Results) — DETERMINISTA ───────
def nbr_ktsp(D, n, k, home=HOME, cand_window=0):
    """
    DCIH — Nicolás Barrientos Results. Heurística propia para el k-TSP, simple,
    explicable y DETERMINISTA (sin azar; mismos datos => mismo tour).

    Fase 1: selección + orden por costo marginal de inserción mínimo (vectorizado),
            empezando en la ciudad-hogar, hasta tener exactamente k nodos.
            Empates resueltos por índice de nodo => resultado único.
    Fase 2: refinamiento local determinista (2-opt + intercambio dentro/fuera).

    cand_window>0 restringe los candidatos a los más cercanos a la hogar para
    acelerar instancias grandes; nunca por debajo de (k-1) para poder llenar k nodos.
    """
    # Candidatos (acotados de forma determinista por cercanía a la hogar)
    pool_all = np.array([v for v in range(n) if v != home])
    if cand_window:
        eff = max(cand_window, k - 1)
        if eff < pool_all.size:
            order = np.argsort(D[home, pool_all])
            pool_all = pool_all[order[:eff]]
    pool = set(int(v) for v in pool_all)

    # Fase 1 — construcción por inserción más barata, determinista
    second = min(pool, key=lambda v: (D[home, v], v))
    tour = [home, second]; pool.discard(second)
    while len(tour) < k and pool:
        cand = np.array(sorted(pool))           # orden fijo => determinista
        arr = np.asarray(tour); nxt = np.roll(arr, -1)
        dab = D[arr, nxt]
        dac = D[np.ix_(cand, arr)]
        dcb = D[np.ix_(cand, nxt)]
        delta = dac + dcb - dab[None, :]        # (|cand|, m)
        # desempate determinista: menor delta; si empata, menor índice de nodo, luego posición
        m = delta.shape[1]
        best_ci = best_pos = None; best_delta = None
        # argmin global con desempate estable por (delta, node, pos)
        flat = np.argsort(delta, axis=None, kind="stable")[0]
        ci, ei = divmod(int(flat), m)
        node = int(cand[ci]); pos = ei + 1
        tour.insert(pos, node); pool.discard(node)

    # Fase 2 — refinamiento local determinista (mismo núcleo vectorizado)
    cur = two_opt_cycle(tour, D)
    for _ in range(12):
        cur, ch = swap_in_out(cur, D, n, home)
        cur = two_opt_cycle(cur, D)
        if not ch:
            break

    # Garantía de tamaño: completar de forma determinista si faltara algún nodo
    if len(cur) < k:
        present = set(cur)
        rest = [v for v in range(n) if v not in present]
        rest.sort(key=lambda v: (float(np.min(D[v, cur])), v))
        for v in rest:
            if len(cur) >= k: break
            arr = np.asarray(cur); nxt = np.roll(arr, -1)
            delta = D[v, arr] + D[v, nxt] - D[arr, nxt]
            pos = int(np.argmin(delta)) + 1
            cur.insert(pos, v)
        cur = two_opt_cycle(cur, D)

    return {"tour": cur, "length": tour_length(cur, D),
            "source": "DCIH_Nicolas_Barrientos_Results_deterministic"}

# ─── BINARIO LKH-3 OFICIAL (formato k-TSP: SPECIAL + K) ───────────────────────
# ─── HEURÍSTICAS ADICIONALES: NN, NI, FI, RI (inserción) + SA (metaheurística) ─
def nn_ktsp(D, n, k, home=HOME):
    """Nearest Neighbor: avanza al nodo más cercano no visitado hasta k; LS. Determinista."""
    inset = np.zeros(n, dtype=bool); inset[home] = True
    tour = [home]; current = home
    while len(tour) < k:
        outside = np.where(~inset)[0]
        if outside.size == 0: break
        node = int(outside[int(np.argmin(D[current, outside]))])
        tour.append(node); inset[node] = True; current = node
    tour = local_search_ktsp(tour, D, n, home)
    return {"tour": tour, "length": tour_length(tour, D), "source": "nearest_neighbor+LS"}

def _insertion_build(D, n, k, home, selection, rng=None):
    """Construcción por inserción; selection in {'nearest','farthest','random'}."""
    inset = np.zeros(n, dtype=bool); inset[home] = True
    order = np.argsort(D[home])
    second = next(int(v) for v in order if v != home)
    tour = [home, second]; inset[second] = True
    while len(tour) < k:
        outside = np.where(~inset)[0]
        if outside.size == 0: break
        nearest = D[np.ix_(outside, tour)].min(axis=1)
        if selection == "nearest":    pick = int(np.argmin(nearest))
        elif selection == "farthest": pick = int(np.argmax(nearest))
        else:                         pick = rng.randrange(outside.size)
        node = int(outside[pick])
        arr = np.asarray(tour); nxt = np.roll(arr, -1)
        delta = D[node, arr] + D[node, nxt] - D[arr, nxt]
        pos = int(np.argmin(delta)) + 1
        tour.insert(pos, node); inset[node] = True
    return tour

def ni_ktsp(D, n, k, home=HOME):
    """Nearest Insertion + LS. Determinista."""
    tour = local_search_ktsp(_insertion_build(D, n, k, home, "nearest"), D, n, home)
    return {"tour": tour, "length": tour_length(tour, D), "source": "nearest_insertion+LS"}

def fi_ktsp(D, n, k, home=HOME):
    """Farthest Insertion + LS. Determinista."""
    tour = local_search_ktsp(_insertion_build(D, n, k, home, "farthest"), D, n, home)
    return {"tour": tour, "length": tour_length(tour, D), "source": "farthest_insertion+LS"}

def ri_ktsp(D, n, k, home=HOME, time_limit=GRASP_TIME, seed=SEED):
    """Random Insertion multi-arranque + LS. Estocástico."""
    rng = random.Random(seed + 7)
    best_tour, best_len = None, float("inf")
    t0 = time.time()
    while time.time() - t0 < time_limit:
        tour = local_search_ktsp(_insertion_build(D, n, k, home, "random", rng), D, n, home)
        L = tour_length(tour, D)
        if L < best_len: best_tour, best_len = tour, L
    if best_tour is None:
        best_tour = local_search_ktsp(_insertion_build(D, n, k, home, "random", rng), D, n, home)
        best_len = tour_length(best_tour, D)
    return {"tour": best_tour, "length": best_len, "source": "random_insertion+LS"}

def sa_ktsp(D, n, k, home=HOME, time_limit=GA_TIME, seed=SEED, alpha=0.9995):
    """Simulated Annealing sobre (subconjunto + orden). Estocástico."""
    rng = random.Random(seed + 9)
    cur = two_opt_cycle(_insertion_build(D, n, k, home, "nearest"), D)
    cur_len = tour_length(cur, D)
    best, best_len = list(cur), cur_len
    inset = np.zeros(n, dtype=bool); inset[np.asarray(cur)] = True
    outside = [int(v) for v in range(n) if not inset[v]]
    pos = np.asarray(cur); nxtp = np.roll(pos, -1)
    T = max(1e-9, float(np.mean(D[pos, nxtp])) * 0.5)
    m = len(cur); t0 = time.time()
    while m >= 4 and time.time() - t0 < time_limit:
        if outside and rng.random() < 0.35:
            p = rng.randrange(1, m)
            a = cur[p-1]; b = cur[(p+1) % m]; old = cur[p]
            new = outside[rng.randrange(len(outside))]
            delta = (D[a,new]+D[new,b]) - (D[a,old]+D[old,b])
            if delta < 0 or rng.random() < np.exp(-delta/T):
                cur[p] = new; inset[old] = False; inset[new] = True
                outside.remove(new); outside.append(old); cur_len += delta
        else:
            i = rng.randrange(1, m-1); j = rng.randrange(i+1, m)
            a = cur[i-1]; ti = cur[i]; tj = cur[j]; b = cur[(j+1) % m]
            if a != tj and ti != b:
                delta = (D[a,tj]+D[ti,b]) - (D[a,ti]+D[tj,b])
                if delta < 0 or rng.random() < np.exp(-delta/T):
                    cur[i:j+1] = cur[i:j+1][::-1]; cur_len += delta
        if cur_len < best_len - 1e-9:
            best_len = cur_len; best = list(cur)
        T *= alpha
        if T < 1e-6:
            T = max(1e-9, float(np.mean(D[np.asarray(cur), np.roll(cur,-1)])) * 0.1)
    best = local_search_ktsp(best, D, n, home)
    return {"tour": best, "length": tour_length(best, D), "source": "simulated_annealing"}

def write_lkh_ktsp_files(name, coords, ewtype, k, workdir: Path):
    """Genera .tsp y .par para el k-TSP de LKH-3 (keyword K = nº de nodos a visitar)."""
    workdir.mkdir(parents=True, exist_ok=True)
    tsp_path  = workdir / f"{name}.tsp"
    par_path  = workdir / f"{name}.par"
    tour_path = workdir / f"{name}_ktsp.tour"
    with open(tsp_path, "w", encoding="utf-8") as f:
        f.write(f"NAME : {name}\n")
        f.write("TYPE : TSP\n")
        f.write(f"DIMENSION : {len(coords)}\n")
        f.write(f"EDGE_WEIGHT_TYPE : {ewtype}\n")
        f.write("NODE_COORD_SECTION\n")
        for idx, (x, y) in enumerate(coords, start=1):
            f.write(f"{idx} {x} {y}\n")
        f.write("EOF\n")
    with open(par_path, "w", encoding="utf-8") as f:
        # k-TSP oficial de LKH-3 (>=3.0.9): SPECIAL + K = nº de nodos a visitar.
        f.write(f"PROBLEM_FILE = {tsp_path}\n")
        f.write("SPECIAL\n")               # usa los movimientos especiales (move type 5 SPECIAL + patching)
        f.write(f"K = {k}\n")               # K = nº de nodos a visitar (incluida la hogar/depot)
        f.write(f"DEPOT = {HOME + 1}\n")     # LKH es 1-based
        f.write(f"RUNS = {LKH3_RUNS}\n")
        f.write(f"MAX_TRIALS = {LKH3_MAX_TRIALS}\n")
        f.write(f"SEED = {SEED}\n")
        f.write(f"OUTPUT_TOUR_FILE = {tour_path}\n")
        f.write("TRACE_LEVEL = 1\n")
    return tsp_path, par_path, tour_path

def parse_lkh_cost_from_log(log_text: str) -> Optional[float]:
    """
    Lee el costo óptimo que LKH imprime, sin recalcularlo nosotros.
    LKH escribe líneas como 'Cost.min = 12345, Cost.avg = ...' y/o
    'Best Cost = 12345'. Tomamos el menor costo reportado.
    """
    import re
    costs = []
    for m in re.finditer(r"Cost\.min\s*=\s*([0-9]+(?:\.[0-9]+)?)", log_text):
        costs.append(float(m.group(1)))
    for m in re.finditer(r"Best\s+Cost\s*=\s*([0-9]+(?:\.[0-9]+)?)", log_text, re.IGNORECASE):
        costs.append(float(m.group(1)))
    # algunas versiones imprimen 'Cost = NNN' en la línea de la mejor solución
    for m in re.finditer(r"^\s*Cost\s*=\s*([0-9]+(?:\.[0-9]+)?)", log_text, re.MULTILINE):
        costs.append(float(m.group(1)))
    return min(costs) if costs else None

def parse_lkh_ktsp_tour(tour_path: Path, n: int, k: int, home=HOME) -> Optional[List[int]]:
    """
    Lee el OUTPUT_TOUR_FILE. Para el k-TSP, LKH lista los nodos del recorrido;
    los nodos NO visitados aparecen como ciclos triviales/aislados o no aparecen
    en el ciclo principal. Extraemos el ciclo que contiene la hogar y nos quedamos
    con sus k nodos. Devolvemos el tour 0-based empezando en la hogar, o None.
    """
    if not tour_path.exists():
        return None
    nums, in_sec = [], False
    for line in tour_path.read_text(errors="ignore").splitlines():
        s = line.strip()
        if s.upper().startswith("TOUR_SECTION"):
            in_sec = True; continue
        if not in_sec:
            continue
        if s == "-1" or s.upper().startswith("EOF"):
            break
        for tok in s.split():
            try:
                v = int(tok) - 1   # a 0-based
                if 0 <= v < n:
                    nums.append(v)
            except Exception:
                pass
    if not nums:
        return None
    # El tour de LKH para k-TSP recorre exactamente los k nodos seleccionados
    # (los n-k restantes no forman parte del ciclo). Si vienen más de k por el
    # formato interno, nos quedamos con la subsecuencia única preservando orden.
    seen, ordered = set(), []
    for v in nums:
        if v not in seen:
            seen.add(v); ordered.append(v)
    # rotar para empezar en la hogar
    if home in ordered:
        h = ordered.index(home); ordered = ordered[h:] + ordered[:h]
    return ordered

def run_lkh3_ktsp(name, coords, D, ewtype, n, k, home=HOME):
    """
    Ejecuta el binario LKH-3 en modo k-TSP. El costo se TOMA DEL LOG de LKH
    (valor que LKH optimizó), no se recalcula, para evitar errores de parsing.
    El tour se lee solo para validar que tenga k nodos con la hogar incluida.
    """
    exe = Path(LKH_EXECUTABLE) if LKH_EXECUTABLE else None
    if not (exe and exe.exists()):
        return None
    workdir = RESULTS_DIR / "lkh3_runs" / name
    _, par_path, tour_path = write_lkh_ktsp_files(name, coords, ewtype, k, workdir)
    log_path = workdir / f"{name}_ktsp.log"
    try:
        timeout = max(120, int(n * 0.8))
        with open(log_path, "w", encoding="utf-8") as log:
            subprocess.run([str(exe), str(par_path)], stdout=log,
                           stderr=subprocess.STDOUT, timeout=timeout)
        log_text = log_path.read_text(errors="ignore")

        # 1) costo que LKH reporta (autoritativo para el k-TSP)
        lkh_cost = parse_lkh_cost_from_log(log_text)

        # 2) tour, solo para validar / mostrar
        tour = parse_lkh_ktsp_tour(tour_path, n, k, home)

        # Validación de coherencia: el tour debe tener k nodos con la hogar.
        tour_ok = False
        recomputed = None
        if tour is not None and len(tour) == k:
            ok, _ = validate_ktsp(tour, n, k, home)
            if ok:
                tour_ok = True
                recomputed = tour_length(tour, D)

        # Elegir el costo a reportar:
        #  - si LKH dio costo y el tour valida y ambos coinciden razonablemente -> costo LKH
        #  - si LKH dio costo pero el tour no valida -> reportar costo LKH igualmente, marcando
        #  - si no hay costo en el log pero el tour valida -> usar el recomputado
        if lkh_cost is not None:
            length = lkh_cost
            note = "log_cost"
            if recomputed is not None and abs(recomputed - lkh_cost) > 1e-6 * max(1.0, lkh_cost):
                note = "log_cost(tour_mismatch)"
        elif recomputed is not None:
            length = recomputed
            note = "recomputed_from_tour"
        else:
            print(f"  [WARN] LKH-3 sin costo legible en {name}; se omite.")
            return None

        if not tour_ok and lkh_cost is not None:
            # No pudimos validar el tour, pero el costo del log es del k-TSP que LKH resolvió.
            print(f"  [INFO] LKH-3 {name}: usando costo del log ({lkh_cost}); "
                  f"tour no validado por formato de salida.")

        return {"tour": (tour if tour_ok else None),
                "length": float(length),
                "source": "LKH-3_official_binary",
                "note": note, "log": str(log_path)}
    except Exception as e:
        print(f"  [WARN] LKH-3 falló en {name}: {e}")
        return None

# ─── MAIN ─────────────────────────────────────────────────────────────────────
def process_instance(name, n_exp, quartile):
    """Procesa UNA instancia para UN cuartil dado. Devuelve (row, tours) o (None, None)."""
    global K_QUARTILE
    K_QUARTILE = quartile  # choose_k y official_k_and_bks leen este valor
    try:
        tsp_file, src = download_tsplib_instance(name)
        coords, ewtype, meta = parse_tsplib_tsp(tsp_file)
        D = tsplib_distance_matrix(coords, ewtype)
    except Exception as e:
        print(f"[ERROR] {name}: {e}")
        return None, None
    n = len(coords)
    k = choose_k(name, n)
    print(f"  [Q{quartile}] n={n}, k={k} (visitar {k} de {n}), EWT={ewtype}")

    gr  = greedy_ktsp(D, n, k)
    gra = grasp_ktsp(D, n, k)
    ga  = ga_ktsp(D, n, k)
    nbr_window = 0 if n <= 200 else min(n - 1, max(120, k + 20))
    nbr = nbr_ktsp(D, n, k, cand_window=nbr_window)
    nn  = nn_ktsp(D, n, k)
    ni  = ni_ktsp(D, n, k)
    fi  = fi_ktsp(D, n, k)
    ri  = ri_ktsp(D, n, k)
    sa  = sa_ktsp(D, n, k)
    lkh = run_lkh3_ktsp(name, coords, D, ewtype, n, k)

    for tag, res in [("greedy", gr), ("GRASP", gra), ("GA", ga), ("DCIH", nbr),
                     ("NN", nn), ("NI", ni), ("FI", fi), ("RI", ri), ("SA", sa)] + ([("LKH", lkh)] if lkh else []):
        if res.get("tour") is None:
            continue
        ok, msg = validate_ktsp(res["tour"], n, k)
        if not ok: print(f"  [WARN] {tag} inválido: {msg}")

    heur_best = min([gr, gra, ga, nbr, nn, ni, fi, ri, sa], key=lambda r: r["length"])
    _, bks_off = official_k_and_bks(name, n)
    if K_MODE == "official" and bks_off is not None:
        ref_len, ref_src = float(bks_off), "Helsgaun_official_BKS"
    elif lkh is not None:
        ref_len, ref_src = lkh["length"], "LKH-3_official_binary"
    else:
        ref_len, ref_src = heur_best["length"], "best_heuristic_provisional"

    def gap(v): return None if ref_len <= 0 else 100.0*(v - ref_len)/ref_len
    row = {
        "Instancia": name, "n": n, "k": k, "Quartil": quartile,
        "EDGE_WEIGHT_TYPE": ewtype,
        "TSPLIB_source": src, "TSPLIB_sha256": sha256_file(tsp_file),
        "Greedy_Len": round(gr["length"], 2),  "Greedy_GAP%": gap(gr["length"]),
        "GRASP_Len":  round(gra["length"], 2), "GRASP_GAP%":  gap(gra["length"]),
        "GA_Len":     round(ga["length"], 2),  "GA_GAP%":     gap(ga["length"]),
        "DCIH_Len":    round(nbr["length"], 2), "DCIH_GAP%":    gap(nbr["length"]),
        "NN_Len":     round(nn["length"], 2),  "NN_GAP%":     gap(nn["length"]),
        "NI_Len":     round(ni["length"], 2),  "NI_GAP%":     gap(ni["length"]),
        "FI_Len":     round(fi["length"], 2),  "FI_GAP%":     gap(fi["length"]),
        "RI_Len":     round(ri["length"], 2),  "RI_GAP%":     gap(ri["length"]),
        "SA_Len":     round(sa["length"], 2),  "SA_GAP%":     gap(sa["length"]),
        "LKH_Len":    (round(lkh["length"], 2) if lkh else None),
        "LKH_GAP%":   (gap(lkh["length"]) if lkh else None),
        "BKS_oficial": (float(bks_off) if bks_off is not None else None),
        "Ref_Len": round(ref_len, 2), "Ref_Source": ref_src,
    }
    tours = {"greedy": gr, "GRASP": gra, "GA": ga, "DCIH": nbr,
             "NN": nn, "NI": ni, "FI": fi, "RI": ri, "SA": sa, "LKH": lkh, "D": D}
    best_tag = min([("greedy",gr),("GRASP",gra),("GA",ga),("DCIH",nbr),
                    ("NN",nn),("NI",ni),("FI",fi),("RI",ri),("SA",sa)] + ([("LKH",lkh)] if lkh else []),
                   key=lambda kv: kv[1]["length"])[0]
    print(f"    -> mejor: {best_tag}  (Ref={ref_src}, Ref_Len={round(ref_len,2)})")
    return row, tours

def main():
    random.seed(SEED); np.random.seed(SEED)
    quartiles = K_QUARTILES if K_MODE == "official" else [K_QUARTILE]
    rows = []
    all_tours = {}   # (name, quartile) -> tours
    for name, n_exp in SELECTED:
        print(f"\n=== {name} (n~{n_exp}) ===")
        for q in quartiles:
            row, tours = process_instance(name, n_exp, q)
            if row is None:
                continue
            rows.append(row)
            all_tours[(name, q)] = tours

    if not rows:
        raise RuntimeError("No se ejecutó ninguna instancia. Revisa la conexión.")
    df = pd.DataFrame(rows).sort_values(["Quartil", "n", "Instancia"]).reset_index(drop=True)

    # ── exportar ──
    csv_path = RESULTS_DIR / "resultados_kTSP.csv"
    df.to_csv(csv_path, index=False)
    # un CSV por cuartil, para tablas separadas en el informe
    per_quartile_paths = []
    for q in sorted(df["Quartil"].unique()):
        qp = RESULTS_DIR / f"resultados_kTSP_Q{q}.csv"
        df[df["Quartil"] == q].to_csv(qp, index=False)
        per_quartile_paths.append(str(qp))

    tours_path = RESULTS_DIR / "tours_kTSP.txt"
    with open(tours_path, "w", encoding="utf-8") as f:
        f.write("k-TSP VERDADERO — tours (subconjunto de k nodos, 1 viajante, min distancia)\n")
        f.write(f"Seed={SEED}, ciudad-hogar=0 (0-based). Cuartiles={quartiles}\n\n")
        for _, r in df.iterrows():
            inst = r["Instancia"]; q = r["Quartil"]; T = all_tours[(inst, q)]
            f.write("="*80 + f"\nInstancia: {inst} | Q{q} | n={r['n']} | k={r['k']} | {r['EDGE_WEIGHT_TYPE']}\n")
            f.write(f"Ref_Len={r['Ref_Len']} ({r['Ref_Source']})\n" + "-"*80 + "\n")
            for tag in ["greedy", "GRASP", "GA", "DCIH", "LKH"]:
                res = T[tag]
                if res is None: continue
                f.write(f"[{tag}] source={res['source']} | Len={round(res['length'],2)}\n")
                if res.get("tour") is not None:
                    f.write(f"  tour(0-based, cerrado al inicio)={res['tour']}\n")
                else:
                    f.write(f"  tour: (no reconstruido; costo tomado del log de LKH-3)\n")
            f.write("\n")

    manifest = {
        "problem": "k-TSP (Pandiri & Singh 2020): seleccionar k nodos incl. hogar, 1 tour, min distancia total",
        "DCIH": "Nicolás Barrientos Results — heurística propia, determinista y replicable",
        "k_mode": K_MODE, "quartiles": quartiles,
        "quartile_meaning": {"1": "k~n/4 (TSPLIB_1)", "2": "k~n/2 (TSPLIB_2)", "3": "k~3n/4 (TSPLIB_3)"},
        "reference": "BKS oficial de Helsgaun cuando está tabulado; si no, LKH-3 o mejor heurística",
        "seed": SEED, "home": HOME, "selected": SELECTED,
        "lkh_executable": LKH_EXECUTABLE or None,
        "outputs": {"csv": str(csv_path), "csv_per_quartile": per_quartile_paths, "tours": str(tours_path)},
    }
    (RESULTS_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

    print("\n" + "="*60 + "\nLISTO.")
    print("CSV combinado:", csv_path)
    for qp in per_quartile_paths:
        print("CSV por cuartil:", qp)
    print("Tours:  ", tours_path)
    print("Manifest:", RESULTS_DIR / "manifest.json")
    print("="*60)
    try:
        from IPython.display import display
        cols = ["Instancia","n","k","Quartil","EDGE_WEIGHT_TYPE","Greedy_Len","GRASP_Len","GA_Len",
                "DCIH_Len","LKH_Len","BKS_oficial","Ref_Len","Ref_Source"]
        display(df[[c for c in cols if c in df.columns]])
    except Exception:
        print(df.head())
    if IN_COLAB:
        for p in [csv_path, tours_path, RESULTS_DIR / "manifest.json"] + [Path(x) for x in per_quartile_paths]:
            try: files.download(str(p))
            except Exception: pass
    return df

df_resultados = main()



=== att48 (n~48) ===
  [Q1] n=48, k=12 (visitar 12 de 48), EWT=ATT
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=1925.0)
  [Q2] n=48, k=24 (visitar 24 de 48), EWT=ATT
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=3603.0)
  [Q3] n=48, k=36 (visitar 36 de 48), EWT=ATT
    -> mejor: greedy  (Ref=Helsgaun_official_BKS, Ref_Len=6563.0)

=== eil51 (n~51) ===
  [Q1] n=51, k=12 (visitar 12 de 51), EWT=EUC_2D
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=82.0)
  [Q2] n=51, k=25 (visitar 25 de 51), EWT=EUC_2D
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=175.0)
  [Q3] n=51, k=38 (visitar 38 de 51), EWT=EUC_2D
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=287.0)

=== berlin52 (n~52) ===
  [Q1] n=52, k=13 (visitar 13 de 52), EWT=EUC_2D
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=679.0)
  [Q2] n=52, k=26 (visitar 26 de 52), EWT=EUC_2D
    -> mejor: GA  (Ref=Helsgaun_official_BKS, Ref_Len=1874.0)
  [Q3] n=52, k=39 (visitar 39 de 52), EWT=EUC_2D
    

,Instancia,n,k,Quartil,EDGE_WEIGHT_TYPE,Greedy_Len,GRASP_Len,GA_Len,NBR_Len,LKH_Len,BKS_oficial,Ref_Len,Ref_Source
0,att48,48,12,1,ATT,2363.0,2243.0,1925.0,2363.0,None,1925.0,1925.0,Helsgaun_official_BKS
1,eil51,51,12,1,EUC_2D,83.0,89.0,82.0,83.0,None,82.0,82.0,Helsgaun_official_BKS
2,berlin52,52,13,1,EUC_2D,735.0,792.0,679.0,735.0,None,679.0,679.0,Helsgaun_official_BKS
3,st70,70,17,1,EUC_2D,120.0,120.0,125.0,120.0,None,120.0,120.0,Helsgaun_official_BKS
4,eil76,76,19,1,EUC_2D,129.0,124.0,113.0,129.0,None,102.0,102.0,Helsgaun_official_BKS
...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,lin318,318,238,3,EUC_2D,32123.0,31914.0,32448.0,32784.0,None,25303.0,25303.0,Helsgaun_official_BKS
110,rd400,400,300,3,EUC_2D,11910.0,12063.0,11884.0,12030.0,None,9972.0,9972.0,Helsgaun_official_BKS
111,fl417,417,312,3,EUC_2D,10642.0,11199.0,9646.0,11403.0,None,7930.0,7930.0,Helsgaun_official_BKS
112,pr439,439,329,3,EUC_2D,67815.0,81951.0,82459.0,76775.0,None,58006.0,58006.0,Helsgaun_official_BKS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (4 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Paso final — Tabla comparativa para el informe

Compara las heurísticas (y el binario LKH-3 oficial si lo ejecutaste) con el GAP respecto a la
referencia. Recuerda: aquí el objetivo es **distancia total** de un único tour que visita
**exactamente k nodos**.

In [ ]:
import numpy as np, pandas as pd
assert "df_resultados" in globals(), "Ejecuta primero la celda principal."
_df = df_resultados.copy()
_has_q = "Quartil" in _df.columns

_lkh_used = _df["LKH_Len"].notna().sum() if "LKH_Len" in _df.columns else 0
print(f"[INFO] Filas (instancia x cuartil): {len(_df)}")
print(f"[INFO] Con binario LKH-3 oficial: {int(_lkh_used)}/{len(_df)}")
if "Ref_Source" in _df.columns:
    _noff = int((_df["Ref_Source"] == "Helsgaun_official_BKS").sum())
    print(f"[INFO] Comparadas contra el BKS OFICIAL de Helsgaun: {_noff}/{len(_df)}")
    if _noff > 0:
        print("[INFO] GAP% = 100*(longitud - BKS_oficial)/BKS_oficial.")
        print("       Un GAP negativo sería imposible (BKS es óptimo/near-óptimo).")
        _neg = _df[["Greedy_GAP%","GRASP_GAP%","GA_GAP%","DCIH_GAP%",
                    "NN_GAP%","NI_GAP%","FI_GAP%","RI_GAP%","SA_GAP%"]].lt(-1e-6).sum().sum()
        if _neg: print(f"[ALERTA] Hay {int(_neg)} GAP negativos: revisar esas filas.")
        else:    print("[OK] No hay GAP negativos: resultados coherentes con el BKS.")

def _fmt(x): return "" if (x is None or (isinstance(x,float) and not np.isfinite(x))) else f"{x:+.2f}%"

_cols = {
    "Instancia": _df["Instancia"], "n": _df["n"], "k": _df["k"],
}
if _has_q: _cols["Q"] = _df["Quartil"]
_cols.update({
    "Tipo": _df["EDGE_WEIGHT_TYPE"],
    "Greedy": _df["Greedy_Len"], "Greedy_GAP": _df["Greedy_GAP%"].map(_fmt),
    "GRASP":  _df["GRASP_Len"],  "GRASP_GAP":  _df["GRASP_GAP%"].map(_fmt),
    "GA":     _df["GA_Len"],     "GA_GAP":     _df["GA_GAP%"].map(_fmt),
    "DCIH":    _df["DCIH_Len"],    "DCIH_GAP":    _df["DCIH_GAP%"].map(_fmt),
    "NN":     _df["NN_Len"],     "NN_GAP":     _df["NN_GAP%"].map(_fmt),
    "NI":     _df["NI_Len"],     "NI_GAP":     _df["NI_GAP%"].map(_fmt),
    "FI":     _df["FI_Len"],     "FI_GAP":     _df["FI_GAP%"].map(_fmt),
    "RI":     _df["RI_Len"],     "RI_GAP":     _df["RI_GAP%"].map(_fmt),
    "SA":     _df["SA_Len"],     "SA_GAP":     _df["SA_GAP%"].map(_fmt),
    "LKH3":   _df["LKH_Len"],    "LKH3_GAP":   _df["LKH_GAP%"].map(_fmt),
    "BKS_of": _df["BKS_oficial"] if "BKS_oficial" in _df.columns else None,
    "Ref":    _df["Ref_Len"],    "Ref_fuente": _df["Ref_Source"],
})
_sort = (["Q","n"] if _has_q else ["n"])
comp = pd.DataFrame(_cols).sort_values(_sort).reset_index(drop=True)

def _summ(sub, gapcol):
    g = sub[gapcol].replace([np.inf,-np.inf],np.nan).dropna()
    return {"GAP_medio_%": round(float(g.mean()),2) if len(g) else np.nan,
            "GAP_mediana_%": round(float(g.median()),2) if len(g) else np.nan,
            "GAP_max_%": round(float(g.max()),2) if len(g) else np.nan}

def _resumen(sub):
    r = pd.DataFrame({
        "Greedy": _summ(sub,"Greedy_GAP%"), "GRASP": _summ(sub,"GRASP_GAP%"),
        "GA": _summ(sub,"GA_GAP%"), "DCIH": _summ(sub,"DCIH_GAP%"),
        "NN": _summ(sub,"NN_GAP%"), "NI": _summ(sub,"NI_GAP%"),
        "FI": _summ(sub,"FI_GAP%"), "RI": _summ(sub,"RI_GAP%"), "SA": _summ(sub,"SA_GAP%"),
        **({"LKH-3": _summ(sub,"LKH_GAP%")} if sub["LKH_Len"].notna().any() else {}),
    }).T
    r["n_instancias"] = len(sub)
    return r

# Resumen global y por cuartil
resumen_global = _resumen(_df)
resumenes_q = {}
if _has_q:
    for q in sorted(_df["Quartil"].unique()):
        resumenes_q[q] = _resumen(_df[_df["Quartil"] == q])

# Desempeño de DCIH por cuartil
_heur = ["Greedy_Len", "GRASP_Len", "GA_Len", "DCIH_Len",
         "NN_Len", "NI_Len", "FI_Len", "RI_Len", "SA_Len"]
def _nbr_wins(sub):
    w = 0
    for _, _r in sub.iterrows():
        _vals = {h: _r[h] for h in _heur if _r[h] is not None}
        if _vals and min(_vals, key=_vals.get) == "DCIH_Len":
            w += 1
    return w
print(f"\n[DCIH=Nicolás Barrientos Results] Mejor o empatada entre heurísticas: "
      f"{_nbr_wins(_df)}/{len(_df)} (global).")
if _has_q:
    for q in sorted(_df["Quartil"].unique()):
        sub = _df[_df["Quartil"] == q]
        print(f"   Q{q} (k~n/{ {1:4,2:2,3:'4*3'}[q] if q in (1,2,3) else '?'}): "
              f"DCIH mejor/empatada en {_nbr_wins(sub)}/{len(sub)}")
print("[DCIH] Determinista: re-ejecutar produce exactamente el mismo tour (replicable).")

try:
    from IPython.display import display, Markdown
    display(Markdown("### Tabla comparativa por instancia y cuartil (objetivo: distancia total)"))
    display(comp)
    display(Markdown("### Resumen de GAP vs BKS oficial — GLOBAL"))
    display(resumen_global)
    for q, rq in resumenes_q.items():
        display(Markdown(f"### Resumen de GAP — Cuartil Q{q}"))
        display(rq)
except Exception:
    print(comp.to_string(index=False)); print(); print(resumen_global.to_string())
    for q, rq in resumenes_q.items():
        print(f"\n--- Q{q} ---"); print(rq.to_string())

# ── exportar ──
out_csv = RESULTS_DIR / "comparativa_informe.csv"
out_md  = RESULTS_DIR / "comparativa_informe.md"
comp.to_csv(out_csv, index=False)
with open(out_md, "w", encoding="utf-8") as f:
    f.write("# Comparativa k-TSP (objetivo: distancia total, k nodos, 1 tour)\n\n")
    f.write(f"- Filas (instancia x cuartil): {len(_df)}\n")
    f.write(f"- Con binario LKH-3 oficial: {int(_lkh_used)}/{len(_df)}\n")
    f.write("- Cuartiles: Q1 (k~n/4), Q2 (k~n/2), Q3 (k~3n/4).\n")
    f.write("- k = nº de nodos visitados (incluida la ciudad-hogar).\n")
    f.write("- GAP% = 100*(Len - Ref)/Ref. Ref = BKS oficial de Helsgaun cuando existe.\n\n")
    f.write("## Resultados por instancia y cuartil\n\n"); f.write(comp.to_markdown(index=False))
    f.write("\n\n## Resumen global\n\n"); f.write(resumen_global.to_markdown())
    for q, rq in resumenes_q.items():
        f.write(f"\n\n## Resumen Q{q}\n\n"); f.write(rq.to_markdown())
    f.write("\n")
print(f"\n[OK] {out_csv}\n[OK] {out_md}")
if 'IN_COLAB' in globals() and IN_COLAB:
    for p in [out_csv, out_md]:
        try: files.download(str(p))
        except Exception: pass

[INFO] Filas (instancia x cuartil): 114
[INFO] Con binario LKH-3 oficial: 0/114
[INFO] Comparadas contra el BKS OFICIAL de Helsgaun: 114/114
[INFO] GAP% = 100*(longitud - BKS_oficial)/BKS_oficial.
       Un GAP negativo sería imposible (BKS es óptimo/near-óptimo).
[OK] No hay GAP negativos: resultados coherentes con el BKS.

[NBR=Nicolás Barrientos Results] Mejor o empatada entre heurísticas: 2/114 (global).
   Q1 (k~n/4): NBR mejor/empatada en 1/38
   Q2 (k~n/2): NBR mejor/empatada en 1/38
   Q3 (k~n/4*3): NBR mejor/empatada en 0/38
[NBR] Determinista: re-ejecutar produce exactamente el mismo tour (replicable).


### Tabla comparativa por instancia y cuartil (objetivo: distancia total)

,Instancia,n,k,Q,Tipo,Greedy,Greedy_GAP,GRASP,GRASP_GAP,GA,GA_GAP,NBR,NBR_GAP,LKH3,LKH3_GAP,BKS_of,Ref,Ref_fuente
0,att48,48,12,1,ATT,2363.0,+22.75%,2243.0,+16.52%,1925.0,+0.00%,2363.0,+22.75%,None,,1925.0,1925.0,Helsgaun_official_BKS
1,eil51,51,12,1,EUC_2D,83.0,+1.22%,89.0,+8.54%,82.0,+0.00%,83.0,+1.22%,None,,82.0,82.0,Helsgaun_official_BKS
2,berlin52,52,13,1,EUC_2D,735.0,+8.25%,792.0,+16.64%,679.0,+0.00%,735.0,+8.25%,None,,679.0,679.0,Helsgaun_official_BKS
3,st70,70,17,1,EUC_2D,120.0,+0.00%,120.0,+0.00%,125.0,+4.17%,120.0,+0.00%,None,,120.0,120.0,Helsgaun_official_BKS
4,eil76,76,19,1,EUC_2D,129.0,+26.47%,124.0,+21.57%,113.0,+10.78%,129.0,+26.47%,None,,102.0,102.0,Helsgaun_official_BKS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,lin318,318,238,3,EUC_2D,32123.0,+26.95%,31914.0,+26.13%,32448.0,+28.24%,32784.0,+29.57%,None,,25303.0,25303.0,Helsgaun_official_BKS
110,rd400,400,300,3,EUC_2D,11910.0,+19.43%,12063.0,+20.97%,11884.0,+19.17%,12030.0,+20.64%,None,,9972.0,9972.0,Helsgaun_official_BKS
111,fl417,417,312,3,EUC_2D,10642.0,+34.20%,11199.0,+41.22%,9646.0,+21.64%,11403.0,+43.80%,None,,7930.0,7930.0,Helsgaun_official_BKS
112,pr439,439,329,3,EUC_2D,67815.0,+16.91%,81951.0,+41.28%,82459.0,+42.16%,76775.0,+32.36%,None,,58006.0,58006.0,Helsgaun_official_BKS


### Resumen de GAP vs BKS oficial — GLOBAL

,GAP_medio_%,GAP_mediana_%,GAP_max_%,n_instancias
Greedy,17.69,16.96,64.32,114
GRASP,18.02,16.96,61.36,114
GA,19.17,15.49,205.13,114
NBR,21.16,21.44,66.82,114


### Resumen de GAP — Cuartil Q1

... (6 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>